# SAM ViT-B — DIMER E2E promptable-segmentation fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/tutorials/sam_vit_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fsam--vit--base-ffcc4d?style=flat)](https://huggingface.co/facebook/sam-vit-base) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fsegment--anything-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/segment-anything) [![arXiv](https://img.shields.io/badge/arXiv-2304.02643-b31b1b.svg)](https://arxiv.org/abs/2304.02643)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** promptable image segmentation (one point click or one box → one object's mask) and bounded supervised fine-tuning of the mask decoder on labelled point/box → mask records, using the pinned `facebook/sam-vit-base` weights (SAM v1, ViT-B)

**This notebook is standalone.** It carries the repository's package (3 modules under `src/sam_vit_segmentation_pipeline/`, at revision `ddb0cab19be3`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `70c1a07f894ebb5b307fd9eaaee97b9dfc16068f` (~375 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `facebook/sam-vit-base` snapshot (a 375 MB `model.safetensors`; no pickle is opened anywhere), fetches the first three row groups of the ADE20K validation parquet shard from the Hugging Face Hub at an immutable revision (about 14 MB over HTTPS range requests; the shard's declared size is checked first and each row group's decoded content is refused on any SHA-256 or byte-total mismatch), turns the 300 images into 296 labelled targets with an interior click and a tight box each, splits them by image into 180 / 45 / 70, segments a synthetic scene through the inference contract with an input manifest and a rejection probe, measures the frozen model's point-prompt and box-prompt IoU over the 70 held-out targets beside two prompt-only baselines, runs a bounded fine-tuning of the mask decoder with the BCE + soft-Dice loss under mixed prompts and validation-IoU epoch selection, scores the held-out targets again per class, re-segments the scene and four held-out targets with the adapted decoder, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify mask parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On an RTX 5070 Ti the whole path took under ten minutes after the downloads; on CPU the image encoder alone costs several seconds per image at the fixed 1024×1024 working size, so expect an hour or more on a 2-vCPU hosted runtime — a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of images with a `<stem>_mask.png` beside each (white = the object; optionally a `labels.csv` of `id`, `file`, `category`) — at least eight images with sides between 16 and 4096 px. The click and the box are derived from each mask (its distance-transform maximum and its tight box), then the records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the ADE20K sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`facebook/sam-vit-base` is the Segment Anything Model of Kirillov et al. (2023) in its smallest published size: a ViT-B image encoder that embeds a 1024×1024 padded image once, a prompt encoder for clicks and boxes, and a lightweight two-way-transformer mask decoder that turns the two embeddings into up to three candidate masks at 256×256 with a predicted IoU each (93,735,472 parameters, of which the decoder is 4,058,340; published under the **Apache-2.0** licence). The pipeline up-samples the chosen candidate to the input resolution, strips the padding and binarises at logit 0; the predicted IoU is a learned, uncalibrated ranking score, not a probability.

What this notebook adds to inference is **adaptation with labelled targets**. SAM was trained to return *some* object at a click — a part, the whole, or the object with its surroundings — which is exactly the ambiguity a scene-parsing dataset resolves one way: ADE20K's targets are whole semantic regions, and in the validation images the qualifying ones are mostly amorphous *stuff* (walls, sky, floors, roads, ceilings) rather than crisp things. On such targets the frozen model's single click is weak — the build record measured a mean IoU of **0.564** for the point prompt on the 70 held-out targets, *below* the 0.614 of simply filling the box — while the box prompt reaches 0.784. So the honest question is narrow: does a bounded fine-tuning of the mask decoder alone (the encoders stay frozen and every training image is embedded once) on 180 targets under **mixed** point and box prompts move the held-out **point-prompt IoU** past the two prompt-only baselines while keeping the box prompt where it was? Nothing here is a claim about your images: it is one seeded split of one sample of one dataset's labelling convention.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 4-file manifest); the upstream `pytorch_model.bin` and TensorFlow weights are not in the manifest and are never staged or loaded. Section 3 stages and digest-verifies those files before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned slice of a real segmentation dataset, turn its annotations into prompt-to-mask targets under a stated selection rule, validate them and split them by image without leakage; segment a synthetic scene through the public API and read the output contract correctly (three candidates, an uncalibrated predicted IoU that can exceed 1.0, one object per call); measure the frozen model's point- and box-prompt IoU beside two prompt-only baselines and read the per-class breakdown; run a bounded fine-tuning of the mask decoder with a stated loss, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; look at the adapted masks next to the frozen ones and the targets; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** automatic "segment everything" mask generation, text prompts (see the sibling Grounding DINO pipeline), mask-input prompts, several objects in one call, semantic class prediction (the class name is carried as a label for the breakdown, never predicted), video, fine-tuning of the image or prompt encoder, evaluation on the full ADE20K validation set or any benchmark proper (only one seeded 296-target sample from its first 300 images is scored here), and any claim that scene-parsing regions stand in for your objects. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. The image encoder's cost is fixed by the 1024×1024 working size (about 3 s per image on the build workstation's CPU, 0.17 s on an RTX 5070 Ti), and the default path encodes every image several times (two frozen evaluations, one cached embedding pass for training, per-epoch validation, two adapted evaluations): the build record measured 30.6 s for the 180 training embeddings and 194 s for the six epochs with validation scoring on the RTX 5070 Ti, and the whole default path took about eight minutes there with the snapshot and row groups already cached. A CPU-only or 2-vCPU hosted runtime will take an hour or more. The pinned `torch==2.14.0` install and the 375 MB checkpoint are the large downloads of the run; the three row groups are about 14 MB.
- **Knowledge:** basic Python, NumPy and PIL; what a binary mask and intersection-over-union are; why a self-made reference is a plumbing check and a held-out split under one dataset's labelling convention is a measurement of that convention only.
- **Data contract:** records are `{id, image, mask, point, box}` — `image` a PIL image (or a file decodable by Pillow) with sides within 16..4096 px, `mask` a boolean array of the same height and width (or a mask image, white = target), `point` one `[x, y]` click inside the mask, `box` the `[x0, y0, x1, y1]` box enclosing it, an optional `category` of at most 32 characters. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; splitting de-duplicates by decoded pixels so no image lands in two splits. BYOD accepts one zip (or directory) of images with a `<stem>_mask.*` each plus an optional `labels.csv`; the notebook derives the click and the box itself.
- **Validation is structural, not semantic:** every image and mask is decoded, the click is checked to lie inside the mask and the box to enclose it, but nothing checks that the mask outlines what you meant — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads three row groups of `scene_parsing/validation/0000.parquet` from `https://huggingface.co/datasets/zhoubolei/scene_parse_150` at the immutable parquet-conversion revision `e660d866…` (about 14 MB over HTTPS range requests; the shard's declared size and every row group's decoded SHA-256 and byte total are pinned in the carried `samples.py` and refused on any mismatch). ADE20K scene parsing is published under the BSD-3-Clause licence (MIT CSAIL, Zhou et al. 2017); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/sam-vit-base` snapshot (~375 MB in total) at revision `70c1a07f894e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'scipy==1.18.1',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'sam-vit-segmentation-pipeline',
    'repository_revision': 'ddb0cab19be364df5c2523f2727767a0228ae557',
    'embedded_module': 'src/sam_vit_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/sam_vit_segmentation_pipeline/pipeline.py', 'src/sam_vit_segmentation_pipeline/metrics.py', 'src/sam_vit_segmentation_pipeline/samples.py'],
    'module_sha256': 'a343d70626301d3d0eaf4927e40f6475b801a18e533c19fa36b3c682ce37b87e',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/sam_vit_segmentation_pipeline/` @ `ddb0cab19be3`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/sam_vit_segmentation_pipeline/pipeline.py`

In [ ]:
"""Promptable image segmentation over the pinned ``facebook/sam-vit-base`` (SAM v1, ViT-B) checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``segment``: one object per call from point
clicks and/or one box, returning boolean masks at input resolution plus the model's predicted IoU per mask.
"""

from __future__ import annotations

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width
import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "facebook/sam-vit-base"
MODEL_REVISION = "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "sam-vit-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Input ceilings. The processor resizes the longest edge to 1024 and pads to 1024x1024 (see
# preprocessor_config.json), so model cost is fixed; the caller's resolution only sets the size of the
# up-sampled output masks. Prompts are one object per call: up to MAX_PROMPTS point clicks (label 1 =
# foreground, 0 = background) and/or one xyxy box.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
NUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs
MASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)
PARAMETER_COUNT = 93_735_472
MASK_DECODER_PARAMETERS = 4_058_340
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
WEIGHT_FILE = "model.safetensors"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
PROMPT_KINDS = ("point", "box", "mixed")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")
    if a.dtype != np.bool_ or b.dtype != np.bool_:
        raise TypeError("mask_iou expects boolean arrays")
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_prompts(
    width: int,
    height: int,
    points: Sequence[Sequence[float]] | None,
    point_labels: Sequence[int] | None,
    box: Sequence[float] | None,
) -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:
    """Check one object's prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""
    if points is None and box is None:
        raise ValueError("provide at least one of points or box")
    clean_points = clean_labels = None
    if points is not None:
        if isinstance(points, str) or not isinstance(points, Sequence):
            raise TypeError("points must be a sequence of [x, y] pairs")
        if not 1 <= len(points) <= MAX_PROMPTS:
            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
        if point_labels is None or len(point_labels) != len(points):
            raise ValueError("point_labels must be given with one 0/1 entry per point")
        clean_points, clean_labels = [], []
        for (x, y), label in zip(points, point_labels, strict=True):
            if not (0 <= x < width and 0 <= y < height):
                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")
            if label not in (0, 1) or isinstance(label, bool):
                raise ValueError(f"point label must be 0 or 1, got {label!r}")
            clean_points.append([float(x), float(y)])
            clean_labels.append(int(label))
    elif point_labels is not None:
        raise ValueError("point_labels given without points")
    clean_box = None
    if box is not None:
        if len(box) != 4:
            raise ValueError("box must be [x0, y0, x1, y1]")
        x0, y0, x1, y1 = (float(v) for v in box)
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")
        clean_box = [x0, y0, x1, y1]
    return clean_points, clean_labels, clean_box


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one object's prompts: point clicks "
        "and/or one xyxy box"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "points": [1, MAX_PROMPTS],
    "point_labels": "one per point, 1 = foreground and 0 = background",
    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",
    "objects_per_call": 1,
    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,
    "preprocessing": (
        "image converted to RGB; the processor resizes the longest edge to 1024 and pads to 1024x1024; "
        "returned masks are up-sampled "
        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"
    ),
}


def _check_inputs(
    image: Any,
    points: Any,
    point_labels: Any,
    box: Any,
    multimask: Any,
) -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    clean_points, clean_labels, clean_box = validate_prompts(
        rgb.width, rgb.height, points, point_labels, box
    )
    if not isinstance(multimask, bool):
        raise TypeError("multimask must be a bool")
    return rgb, clean_points, clean_labels, clean_box


def validate_inputs(
    image: Image.Image,
    *,
    points: Sequence[Sequence[float]] | None = None,
    point_labels: Sequence[int] | None = None,
    box: Sequence[float] | None = None,
    multimask: bool = True,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, clean_points, clean_labels, clean_box = _check_inputs(
        image, points, point_labels, box, multimask
    )
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_points": 0 if clean_points is None else len(clean_points),
                "has_box": clean_box is not None,
            }
        ],
        "points": clean_points,
        "point_labels": clean_labels,
        "box": clean_box,
        "multimask": multimask,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_mask: Any = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one
    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    masks = np.asarray(result["masks"])
    scores = [float(v) for v in result["iou_scores"]]
    best = int(np.argmax(scores)) if scores else None
    base = {
        "task": "promptable single-object image segmentation (point and/or box prompts)",
        "decision_rule": (
            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "
            "acceptance threshold and does not choose for the caller"
        ),
        "score_semantics": (
            "iou_scores are the model's own uncalibrated predicted IoU for each candidate, not a "
            "measured overlap and not a probability; the regression head is unclipped, so a value "
            "may exceed 1.0"
        ),
        "sample_kind": sample_kind,
        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,
        "best_candidate": best,
        "iou_scores_model_predicted": scores,
        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_mask is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth mask was supplied for the evaluated image",
            "needs": (
                "hand-labelled boolean masks for the prompted objects on your own images, scored with "
                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "
                "mask set ships with this repository"
            ),
        }
    reference = np.asarray(reference_mask)
    return {
        **base,
        "metrics": [
            {
                "id": "mask_iou",
                "candidate": index,
                "value": mask_iou(masks[index], reference),
                "selected": index == best,
                "estimation": "one reference mask on a single scene, no dispersion estimate",
            }
            for index in range(masks.shape[0])
        ],
        "reference_area_px": int(reference.sum()),
        "verdict": "sample-sanity",
        "reason": (
            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "
            "benchmark"
        ),
        "needs": (
            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"
        ),
    }


def _trainable_names(model: Any) -> list[str]:
    """Every parameter of the mask decoder (transformer, upscaling and IoU-prediction heads). The image encoder
    and the prompt encoder stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith("mask_decoder.")]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith("mask_decoder.") for n in names):
        raise ValueError("artifact tensors must all belong to the mask decoder")
    if (manifest.get("adapter") or {}).get("prompts") not in PROMPT_KINDS:
        raise ValueError(f"artifact adapter.prompts must be one of {PROMPT_KINDS}")


def _low_res_target(mask: np.ndarray) -> Any:
    """A record's boolean mask in the model's low-resolution frame: resized so the longest side is 1024, padded to
    1024x1024 at the bottom/right (as the processor pads the image), then downsampled to the 256x256 decoder grid."""
    import torch

    target = torch.from_numpy(np.asarray(mask, dtype=np.float32))[None, None]
    height, width = target.shape[2], target.shape[3]
    scale = 1024 / max(height, width)
    new_h, new_w = int(round(height * scale)), int(round(width * scale))
    resized = torch.nn.functional.interpolate(target, size=(new_h, new_w), mode="bilinear", align_corners=False)
    canvas = torch.zeros(1, 1, 1024, 1024)
    canvas[:, :, :new_h, :new_w] = resized
    return torch.nn.functional.interpolate(canvas, size=(256, 256), mode="bilinear", align_corners=False)[0]


@dataclass
class SAMViTSegmentationPipeline:
    """Promptable image segmentation (points/box -> masks) over the pinned SAM ViT-B checkpoint.

    Prompted mode only (`SamModel` + `SamProcessor`); automatic grid-prompt mask generation is not exposed."""

    _runner: Callable[..., tuple[np.ndarray, list[float]]]
    device: str
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SAMViTSegmentationPipeline:
        import torch
        from transformers import SamModel, SamProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        weight_sha256 = None
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            with open(root / MANIFEST_NAME, encoding="utf-8") as handle:
                entries = json.load(handle).get("files", [])
            weight_sha256 = next((e["sha256"] for e in entries if e["path"] == WEIGHT_FILE), None)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        processor = SamProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = SamModel.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:
            prompt_kwargs: dict[str, Any] = {}
            if points is not None:
                prompt_kwargs["input_points"] = [[points]]
                prompt_kwargs["input_labels"] = [[labels]]
            if box is not None:
                prompt_kwargs["input_boxes"] = [[box]]
            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs, multimask_output=multimask)
            # SAM v1 pads the resized image to 1024x1024; reshaped_input_sizes lets post-processing strip it.
            masks = processor.post_process_masks(
                outputs.pred_masks.cpu(),
                inputs["original_sizes"].cpu(),
                inputs["reshaped_input_sizes"].cpu(),
                mask_threshold=MASK_THRESHOLD,
            )[0]
            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]

        return cls(runner, resolved_device, model, processor, weight_sha256)

    def segment(
        self,
        image: Image.Image,
        *,
        points: Sequence[Sequence[float]] | None = None,
        point_labels: Sequence[int] | None = None,
        box: Sequence[float] | None = None,
        multimask: bool = True,
    ) -> dict[str, Any]:
        """Segment one object; returns K boolean masks (K = 3 with multimask, else 1) at input resolution."""
        rgb, clean_points, clean_labels, clean_box = _check_inputs(
            image, points, point_labels, box, multimask
        )
        masks, iou_scores = self._runner(rgb, clean_points, clean_labels, clean_box, multimask)
        masks = np.asarray(masks)
        expected = (NUM_MULTIMASK_OUTPUTS if multimask else 1, rgb.height, rgb.width)
        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:
            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")
        return {
            "masks": masks,
            "iou_scores": [float(v) for v in iou_scores],
            "multimask": multimask,
            "points": clean_points,
            "point_labels": clean_labels,
            "box": clean_box,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for evaluate/adapt")
        return self._model, self._processor


    def _prompt_kwargs(self, record: Mapping[str, Any], prompt: str) -> dict[str, Any]:
        if prompt == "point":
            return {"points": [record["point"]], "point_labels": [1]}
        if prompt == "box":
            return {"box": record["box"]}
        raise ValueError(f"prompt must be 'point' or 'box', got {prompt!r}")


    def predict_mask(self, record: Mapping[str, Any], *, prompt: str = "point") -> np.ndarray:
        """One boolean mask for a record: `segment` with the record's point (label 1) or box, keeping the multimask
        output the model itself scores highest — the single-mask policy the corpus measures use."""
        result = self.segment(record["image"], multimask=True, **self._prompt_kwargs(record, prompt))
        return result["masks"][int(np.argmax(result["iou_scores"]))]


    def evaluate(self, records: Sequence[Mapping[str, Any]], *, prompt: str = "point", progress: Callable[[int, int], None] | None = None) -> dict[str, Any]:
        """Segment every validated record from one prompt kind and score the masks with `metrics.segmentation_metrics`
        (mean IoU and hit rate, overall and per category)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import segmentation_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if prompt not in ("point", "box"):
            raise ValueError("prompt must be 'point' or 'box'")
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        masks = []
        for i, record in enumerate(checked):
            masks.append(self.predict_mask(record, prompt=prompt))
            if progress is not None:
                progress(i + 1, len(checked))
        metrics = segmentation_metrics(masks, checked)
        metrics.update(
            {
                "prompt": prompt,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics


    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 5e-5,
        prompts: str = "mixed",
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the mask decoder only: the frozen image encoder embeds every training image once
        (cached), the frozen prompt encoder embeds the record's point or box (`prompts`: 'point', 'box' or 'mixed' —
        a seeded coin per record and epoch), and the decoder's single-mask logits are trained against the target mask
        in the 256x256 low-resolution frame with binary cross-entropy plus a soft Dice term. AdamW (no weight decay),
        gradient clipping at 1.0, one record per step, seeded shuffling, no scheduler. Epoch 0 records the frozen
        model's validation metrics; the epoch with the highest validation point-prompt IoU is kept (the final one
        without a validation split). On any exception the frozen weights are restored."""
        model, processor = self._require_model()  # refuse before importing torch
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if prompts not in PROMPT_KINDS:
            raise ValueError(f"prompts must be one of {PROMPT_KINDS}")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model)
        name_set = set(names)
        device = torch.device(self.device)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_adapter = self.adapter
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked, prompt="point")
            return {k: result[k] for k in ("iou", "hit_rate", "n")}

        try:
            for param in model.parameters():
                param.requires_grad_(False)
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            embed_started = time.perf_counter()
            embeddings: dict[str, Any] = {}
            with torch.no_grad():  # not inference_mode: the cached embeddings feed a backward pass
                for record in train_checked:
                    pixel_values = processor(images=record["image"], return_tensors="pt")["pixel_values"].to(device)
                    embeddings[record["id"]] = model.get_image_embeddings(pixel_values).clone()
            embed_seconds = round(time.perf_counter() - embed_started, 3)
            targets = {record["id"]: _low_res_target(record["mask"]).to(device) for record in train_checked}
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("iou", -1.0)
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(train_checked)
                rng.shuffle(order)
                losses = []
                for record in order:
                    use_box = prompts == "box" or (prompts == "mixed" and rng.random() < 0.5)
                    if use_box:
                        encoded = processor(images=record["image"], input_boxes=[[record["box"]]], return_tensors="pt")
                        output = model(image_embeddings=embeddings[record["id"]], input_boxes=encoded["input_boxes"].to(device), multimask_output=False)
                    else:
                        encoded = processor(images=record["image"], input_points=[[[record["point"]]]], input_labels=[[[1]]], return_tensors="pt")
                        output = model(image_embeddings=embeddings[record["id"]], input_points=encoded["input_points"].to(device), input_labels=encoded["input_labels"].to(device), multimask_output=False)
                    logits = output.pred_masks[0, 0]
                    target = targets[record["id"]]
                    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, target)
                    prob = torch.sigmoid(logits)
                    dice = 1.0 - (2.0 * (prob * target).sum() + 1.0) / (prob.sum() + target.sum() + 1.0)
                    loss = bce + dice
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["iou"] > best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("iou", -1.0)
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        self.adapter = {
            "prompts": prompts,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation point-prompt IoU" if val_checked is not None else "final epoch (no validation split)",
            "loss": "binary cross-entropy + soft Dice on the 256x256 single-mask logits",
            "lr": float(lr),
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "embedding_seconds": embed_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)


    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained mask-decoder tensors as safetensors plus a manifest naming the base, the digests and the
        training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHT_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out


    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the mask decoder."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)


    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SAMViTSegmentationPipeline:
        """Load the verified base snapshot, then overlay the adapter (verified before deserialising)."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/sam_vit_segmentation_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level prompt-segmentation measures and two non-neural baselines, in numpy.

``segmentation_metrics`` scores one predicted boolean mask per record against ``record['mask']``: mean IoU
(``pipeline.mask_iou``) and the fraction of records with IoU at least 0.5 (``hit_rate``), overall and per
``category``. The baselines answer from the prompts alone — the filled box, or a disk around the point with the
target's area — and are scored by the same function.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .pipeline import mask_iou` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "iou": "mean intersection-over-union between the predicted mask and the record's target mask; in 0..1, higher is better",
    "hit_rate": "fraction of records whose predicted mask reaches IoU >= 0.5 with the target; in 0..1",
}
HIT_THRESHOLD = 0.5


def segmentation_metrics(masks: Sequence[np.ndarray], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one boolean mask per record against `record['mask']`; per-record rows, means overall and per category.
    Raises when the lengths differ or nothing is scored."""
    if len(masks) != len(records) or not masks:
        raise ValueError("masks and records must be non-empty and the same length")
    rows = []
    for mask, record in zip(masks, records, strict=True):
        iou = mask_iou(np.asarray(mask, dtype=bool), np.asarray(record["mask"], dtype=bool))
        rows.append({"id": record["id"], "category": record.get("category"), "iou": iou, "hit": iou >= HIT_THRESHOLD})

    def _mean(items: Sequence[Mapping[str, Any]]) -> dict[str, float | int]:
        return {"n": len(items), "iou": float(np.mean([r["iou"] for r in items])), "hit_rate": float(np.mean([r["hit"] for r in items]))}

    categories = sorted({r["category"] for r in rows if r["category"] is not None})
    return {
        **_mean(rows),
        "per_category": {c: _mean([r for r in rows if r["category"] == c]) for c in categories},
        "per_record": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def box_fill_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """The prompt as the answer: every pixel inside the box."""
    masks = []
    for record in records:
        target = np.asarray(record["mask"], dtype=bool)
        mask = np.zeros_like(target)
        x0, y0, x1, y1 = (int(round(v)) for v in record["box"])
        mask[y0 : y1 + 1, x0 : x1 + 1] = True
        masks.append(mask)
    return {**segmentation_metrics(masks, records), "baseline": "the box prompt filled"}


def centre_disk_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """A disk around the point prompt with the target's own area — the size is given away, the shape is not."""
    masks = []
    for record in records:
        target = np.asarray(record["mask"], dtype=bool)
        radius = float(np.sqrt(target.sum() / np.pi))
        yy, xx = np.ogrid[: target.shape[0], : target.shape[1]]
        x, y = record["point"]
        masks.append((yy - y) ** 2 + (xx - x) ** 2 <= radius**2)
    return {**segmentation_metrics(masks, records), "baseline": "a disk around the point prompt with the target's area"}

**Module 3/3:** `src/sam_vit_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled prompt-segmentation datasets for the adaptation contract: the digest-pinned ADE20K sample, the record
contract and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, mask, point, box}`` where ``image`` is a PIL image (sides within the pipeline's ceilings),
``mask`` the boolean target (same height and width), ``point`` one ``[x, y]`` foreground click inside the mask and
``box`` the ``[x0, y0, x1, y1]`` box around it — the two prompts the model is asked to turn into that mask. An
optional ``category`` (free text, at most 32 characters) is carried into the per-category breakdown.

The default sample is drawn from the ADE20K scene-parsing validation set (Zhou et al. 2017/2019, **BSD-3-Clause**)
as converted to parquet by the Hugging Face Hub at an immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups
of the validation shard are read with HTTPS range requests (about 4.8 MB each; the shard's declared size is checked
first and every row group's decoded content is refused unless its SHA-256 matches the pin), and each image yields
one target — the largest connected component of one semantic class covering 3..35 % of the image — with an interior
point (the distance-transform maximum) and the tight box. The class name is the record's ``category``.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, validate_image, validate_prompts` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "ADE20K scene parsing (validation), first three parquet row groups"
CORPUS_REPO = "zhoubolei/scene_parse_150"
CORPUS_REVISION = "e660d866a1351c70bcf07925d2600b60bd6e3bc3"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "scene_parsing/validation/0000.parquet"
CORPUS_SHA256 = "79742deaf2661c4740a75d7c26dd361b50209240f6f25aecd29883b96099f879"  # the whole shard (LFS oid)
CORPUS_BYTES = 89_146_778
CORPUS_ROWS = 2_000
CORPUS_ROW_GROUPS = 3  # of 20; 100 images each
CORPUS_LICENSE = "BSD-3-Clause (MIT CSAIL scene parsing benchmark, Zhou et al. 2017; https://github.com/CSAILVision/sceneparsing)"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated image + annotation bytes of each row group, in row order.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("6a056583387d31c659ef2ae0f10eed4184776ec3f70ec9250e9fb2ecf809e4f0", 4_600_209),
    1: ("020c24dc20b9bf6acbc2092cf94d7db8dc3e21516ec60e02591dd051ead63539", 5_007_242),
    2: ("07c4516da0c55b6d9cceecc534a18930337c52b5dc38e8a8dd559f39136d4c3c", 4_540_332),
}
DEFAULT_CACHE_DIR = Path("weights") / "ade20k"

TARGET_MIN_FRACTION = 0.03  # of the image area
TARGET_MAX_FRACTION = 0.35
SAMPLE_MAX_SIDE = 1024
SAMPLE_MIN_SIDE = 128
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 180, "validation": 45, "test": 70}  # of the ~296 records the three row groups yield
SAMPLE_DIGEST = "509007cac0b005f76294dd93c107ff3b8310b8a7a9ec4b69511641e686b223d0"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_CATEGORY_CHARS = 32
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")

# objectInfo150.csv (CSAILVision/sceneparsing), first name of each row; index 0 is "other objects".
ADE_CLASSES: dict[int, str] = {
    1: "wall", 2: "building", 3: "sky", 4: "floor", 5: "tree", 6: "ceiling", 7: "road", 8: "bed", 9: "windowpane", 10: "grass",
    11: "cabinet", 12: "sidewalk", 13: "person", 14: "earth", 15: "door", 16: "table", 17: "mountain", 18: "plant", 19: "curtain", 20: "chair",
    21: "car", 22: "water", 23: "painting", 24: "sofa", 25: "shelf", 26: "house", 27: "sea", 28: "mirror", 29: "rug", 30: "field",
    31: "armchair", 32: "seat", 33: "fence", 34: "desk", 35: "rock", 36: "wardrobe", 37: "lamp", 38: "bathtub", 39: "railing", 40: "cushion",
    41: "base", 42: "box", 43: "column", 44: "signboard", 45: "chest of drawers", 46: "counter", 47: "sand", 48: "sink", 49: "skyscraper", 50: "fireplace",
    51: "refrigerator", 52: "grandstand", 53: "path", 54: "stairs", 55: "runway", 56: "case", 57: "pool table", 58: "pillow", 59: "screen door", 60: "stairway",
    61: "river", 62: "bridge", 63: "bookcase", 64: "blind", 65: "coffee table", 66: "toilet", 67: "flower", 68: "book", 69: "hill", 70: "bench",
    71: "countertop", 72: "stove", 73: "palm", 74: "kitchen island", 75: "computer", 76: "swivel chair", 77: "boat", 78: "bar", 79: "arcade machine", 80: "hovel",
    81: "bus", 82: "towel", 83: "light", 84: "truck", 85: "tower", 86: "chandelier", 87: "awning", 88: "streetlight", 89: "booth", 90: "television receiver",
    91: "airplane", 92: "dirt track", 93: "apparel", 94: "pole", 95: "land", 96: "bannister", 97: "escalator", 98: "ottoman", 99: "bottle", 100: "buffet",
    101: "poster", 102: "stage", 103: "van", 104: "ship", 105: "fountain", 106: "conveyer belt", 107: "canopy", 108: "washer", 109: "plaything", 110: "swimming pool",
    111: "stool", 112: "barrel", 113: "basket", 114: "waterfall", 115: "tent", 116: "bag", 117: "minibike", 118: "cradle", 119: "oven", 120: "ball",
    121: "food", 122: "step", 123: "tank", 124: "trade name", 125: "microwave", 126: "pot", 127: "animal", 128: "bicycle", 129: "lake", 130: "dishwasher",
    131: "screen", 132: "blanket", 133: "sculpture", 134: "hood", 135: "sconce", 136: "vase", 137: "traffic light", 138: "tray", 139: "ashcan", 140: "fan",
    141: "pier", 142: "crt screen", 143: "plate", 144: "monitor", 145: "bulletin board", 146: "shower", 147: "radiator", 148: "glass", 149: "clock", 150: "flag",
}


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and a few row groups without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "sam-vit-segmentation-pipeline"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "sam-vit-segmentation-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        for key in ("image", "annotation"):
            data = row[key]["bytes"]
            digest.update(data)
            total += len(data)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, bytes]]]:
    """Return the pinned row groups as lists of `{image, annotation}` byte pairs, from the cache (one parquet file per
    row group) or the Hub (footer + the row groups it needs, over range requests). Every row group's decoded content is
    refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's declared size."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, bytes]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"validation-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["image", "annotation"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"image": r["image"]["bytes"], "annotation": r["annotation"]["bytes"]} for r in rows]
    return out


# ---------------------------------------------------------------------------------------------------------
# Targets
# ---------------------------------------------------------------------------------------------------------


def connected_components(mask: np.ndarray) -> tuple[np.ndarray, int]:
    """4-connected component labels of a boolean mask (0 = background, 1..n = components)."""
    from scipy import ndimage

    labels, n = ndimage.label(np.asarray(mask, dtype=bool))
    return labels, int(n)


def interior_point(mask: np.ndarray) -> list[float]:
    """The `[x, y]` pixel of a boolean mask farthest from its boundary (the distance-transform maximum)."""
    from scipy import ndimage

    arr = np.asarray(mask, dtype=bool)
    if not arr.any():
        raise ValueError("mask is empty")
    distance = ndimage.distance_transform_edt(arr)
    y, x = np.unravel_index(int(distance.argmax()), distance.shape)
    return [float(x), float(y)]


def mask_box(mask: np.ndarray) -> list[float]:
    """The tight `[x0, y0, x1, y1]` box around a boolean mask."""
    ys, xs = np.where(np.asarray(mask, dtype=bool))
    if ys.size == 0:
        raise ValueError("mask is empty")
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def select_target(annotation: np.ndarray, *, min_fraction: float = TARGET_MIN_FRACTION, max_fraction: float = TARGET_MAX_FRACTION) -> tuple[np.ndarray, int] | None:
    """The largest 4-connected component of any non-background class whose area is within the fractions, as
    `(mask, class_id)`, or None when no component qualifies."""
    ann = np.asarray(annotation)
    area = ann.size
    best: tuple[np.ndarray, int, int] | None = None
    for cls in np.unique(ann):
        if int(cls) == 0:
            continue
        labels, n = connected_components(ann == cls)
        if n == 0:
            continue
        counts = np.bincount(labels.ravel())[1:]
        k = int(counts.argmax()) + 1
        size = int(counts[k - 1])
        if min_fraction * area <= size <= max_fraction * area and (best is None or size > best[2]):
            best = (labels == k, int(cls), size)
    return None if best is None else (best[0], best[1])


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, bytes]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records: one target per image that has a qualifying component."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            image = Image.open(io.BytesIO(row["image"]))
            image.load()
            if max(image.size) > SAMPLE_MAX_SIDE or min(image.size) < SAMPLE_MIN_SIDE:
                continue
            annotation = np.array(Image.open(io.BytesIO(row["annotation"])))
            if annotation.ndim != 2 or annotation.shape != (image.height, image.width):
                continue
            target = select_target(annotation)
            if target is None:
                continue
            mask, cls = target
            out.append(
                {
                    "id": f"ade-val-{group * 100 + index}",
                    "image": image.convert("RGB"),
                    "mask": mask,
                    "point": interior_point(mask),
                    "box": mask_box(mask),
                    "category": ADE_CLASSES.get(cls, f"class-{cls}"),
                    "ade_class_id": cls,
                    "source_row_group": group,
                }
            )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded image-level draw: shuffle the records and cut `sizes` (train / validation / test) in order."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(r) for r in records]
    random.Random(seed).shuffle(pool)
    needed = sum(sizes.values())
    if len(pool) < needed:
        raise ValueError(f"only {len(pool)} records available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        out[name] = pool[cursor : cursor + count]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def _as_mask(mask: Any, where: str) -> np.ndarray:
    if isinstance(mask, str | Path):
        path = Path(mask)
        if not path.is_file():
            raise ValueError(f"{where}: mask file not found: {path}")
        mask = Image.open(path)
        mask.load()
    if isinstance(mask, Image.Image):
        arr = np.asarray(mask.convert("L")) > 127
    else:
        arr = np.asarray(mask)
        if arr.dtype != np.bool_:
            raise ValueError(f"{where}: mask must be a boolean array or a mask image")
    if arr.ndim != 2:
        raise ValueError(f"{where}: mask must be two-dimensional")
    if not arr.any():
        raise ValueError(f"{where}: mask is empty")
    return arr


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/mask/point/box")
    for key in ("id", "image", "mask", "point", "box"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except TypeError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    mask = _as_mask(record["mask"], f"{where}.mask")
    if mask.shape != (image.height, image.width):
        raise ValueError(f"{where}: mask {mask.shape} does not match image {(image.height, image.width)}")
    try:
        points, _labels, box = validate_prompts(image.width, image.height, [record["point"]], [1], record["box"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    x, y = points[0]
    if not mask[min(int(round(y)), mask.shape[0] - 1), min(int(round(x)), mask.shape[1] - 1)]:
        raise ValueError(f"{where}: point {points[0]} is not inside the mask")
    ys, xs = np.where(mask)
    if not (box[0] <= xs.min() and box[1] <= ys.min() and box[2] >= xs.max() and box[3] >= ys.max()):
        raise ValueError(f"{where}: box {box} does not enclose the mask")
    item = {"id": rid, "image": image, "mask": mask, "point": [float(x), float(y)], "box": [float(v) for v in box]}
    category = record.get("category")
    if category is not None:
        if not isinstance(category, str) or not 1 <= len(category.strip()) <= MAX_CATEGORY_CHARS:
            raise ValueError(f"{where}: category must be a str of 1..{MAX_CATEGORY_CHARS} characters")
        item["category"] = category.strip()
    for key in ("ade_class_id", "source_row_group"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a prompt-segmentation dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, mask, point, box} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids, categories = [], set(), {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if "category" in item:
            categories[item["category"]] = categories.get(item["category"], 0) + 1
        checked.append(item)
    fractions = [float(r["mask"].mean()) for r in checked]
    sides = [max(r["image"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "category_counts": dict(sorted(categories.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "mask_fraction": {"min": round(min(fractions), 4), "max": round(max(fractions), 4)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, mask digest, point, box)."""
    parts = sorted(
        f"{r['id']}:{image_digest(r['image'])}:{_sha256_bytes(np.packbits(np.asarray(r['mask'], dtype=bool)).tobytes())}:{json.dumps(r['point'])}:{json.dumps(r['box'])}"
        for r in records
    )
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding image files and, for each, a `<stem>_mask.png` (white = target)
    with an optional `labels.csv` (`id`, `file`, `category`); the point and box are derived from the mask."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    labels: dict[str, tuple[str, str]] = {}
    if "labels.csv" in members:
        for row in csv.DictReader(io.StringIO(members["labels.csv"].decode("utf-8-sig"))):
            labels[str(row.get("file", "")).strip()] = (str(row.get("id", "")).strip(), str(row.get("category", "")).strip())
    out = []
    for name, data in members.items():
        stem = Path(name).stem
        if name == "labels.csv" or stem.endswith("_mask"):
            continue
        mask_name = next((m for m in members if Path(m).stem == f"{stem}_mask"), None)
        if mask_name is None:
            raise ValueError(f"BYOD image {name} has no {stem}_mask.* file")
        try:
            image = Image.open(io.BytesIO(data))
            image.load()
            mask = np.asarray(Image.open(io.BytesIO(members[mask_name])).convert("L")) > 127
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable image: {name}") from exc
        if not mask.any():
            raise ValueError(f"BYOD mask {mask_name} is empty")
        rid, category = labels.get(name, ("", ""))
        record: dict[str, Any] = {
            "id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", stem)[:64],
            "image": image.convert("RGB"),
            "mask": mask,
            "point": interior_point(mask),
            "box": mask_box(mask),
        }
        if category:
            record["category"] = category
        out.append(record)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, category, image size, mask fraction, point, box, provenance), not the BYOD format."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "category", "width", "height", "mask_fraction", "point_x", "point_y", "x0", "y0", "x1", "y1", "source_row_group"])
        for r in records:
            writer.writerow([r["id"], r.get("category", ""), r["image"].width, r["image"].height, round(float(np.asarray(r["mask"]).mean()), 4), *r["point"], *r["box"], r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `70c1a07f894e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "sam-vit-base",
  "modelId": "facebook/sam-vit-base",
  "revision": "70c1a07f894ebb5b307fd9eaaee97b9dfc16068f",
  "files": [
    {
      "path": "README.md",
      "bytes": 6727,
      "sha256": "7a8579105233ff1e8fdc6c8387e102e88458e94c873463f640c416de01c921dd"
    },
    {
      "path": "config.json",
      "bytes": 6566,
      "sha256": "5ebd0d8643b486f3a716bf17c2a15531eb818b2b96ff0c6c5dcc88fa015161af"
    },
    {
      "path": "model.safetensors",
      "bytes": 374979480,
      "sha256": "892c410e496344e527255ccdcb2cb7244a609acb5389c7c4fdba1288f861c579"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 466,
      "sha256": "225545a743c654e3c495ec6f545a0eaba57c8ba3fbbd8483b3cb1c0fc58db517"
    }
  ],
  "totalBytes": 374993239
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. ADE20K targets, prompts and split

`fetch_corpus` returns the three pinned row groups from the cache under `weights/ade20k/` (one parquet file per row group, re-hashed on every read) or the Hub — the shard's footer and the wanted row groups are read over HTTPS range requests, and each row group's decoded image and annotation bytes are refused unless their SHA-256 and byte total match `ROW_GROUP_PINS`. `read_corpus` turns each image into at most one record under a stated rule: `select_target` keeps the largest 4-connected component of any class covering `TARGET_MIN_FRACTION`..`TARGET_MAX_FRACTION` of the image (3..35 %), the **point** prompt is the mask's distance-transform maximum (`interior_point`, a click well inside the region) and the **box** its tight bounds (`mask_box`); the ADE20K class name is the record's `category`. `build_sample_dataset` draws a seeded image-level split (180 / 45 / 70); `validate_dataset` checks every record against the contract and `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared; the training split's summary table is written to `outputs/sam_vit_segmentation_train.csv`.

Look for: 300 rows → 296 targets over about 55 classes, image sides within 240..975 px, mask fractions within 3..35 %, three digests, and four refusal probes — a duplicate id, a click outside its mask, a box that does not enclose the mask, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/ade20k')
    corpus = read_corpus(corpus_groups)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_REPO}@{CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'rows': sum(len(v) for v in corpus_groups.values()), 'bytes': sum(len(r['image']) + len(r['annotation']) for v in corpus_groups.values() for r in v), 'targets': len(corpus), 'classes': len({r['category'] for r in corpus}), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/sam_vit_segmentation_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'target_rule': {'component': 'largest 4-connected component of one class', 'area_fraction': [TARGET_MIN_FRACTION, TARGET_MAX_FRACTION], 'point': 'distance-transform maximum', 'box': 'tight bounds'}})
for name, manifest in dataset_manifests.items():
    top = dict(sorted(manifest['category_counts'].items(), key=lambda kv: -kv[1])[:6])
    print({name: {'n': manifest['n_records'], 'classes': len(manifest['category_counts']), 'top_classes': top, 'image_side': manifest['image_side'], 'mask_fraction': manifest['mask_fraction'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'category': example.get('category'), 'image': list(example['image'].size), 'mask_px': int(example['mask'].sum()), 'point': example['point'], 'box': example['box']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'click outside its mask': [{**train_records[0], 'point': [0.0, 0.0]}, *train_records[1:8]],
    'box does not enclose the mask': [{**train_records[0], 'box': [train_records[0]['point'][0], train_records[0]['point'][1], train_records[0]['point'][0] + 1, train_records[0]['point'][1] + 1]}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Segment a synthetic scene through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a deterministic 320×240 RGB scene drawn in code — grey background, a dark filled rectangle at `[40, 60, 140, 180]` and a red filled disc at `[200, 80, 280, 160]` — with one foreground click at (90, 120) inside the rectangle and the rectangle kept as a reference mask; a different image family from the photographs, and a scene the adapted decoder will segment again in Section 9. `validate_inputs` applies exactly the checks `segment` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, 1..`MAX_PROMPTS` clicks inside the image with 0/1 labels, one xyxy box) and returns an input manifest; a click outside the image is validated too and its rejection recorded as a finding. `segment` returns `masks` of shape `(3, H, W)` with `multimask=True`, one **model-predicted** IoU per candidate — a learned, uncalibrated ranking score that **can exceed 1.0** — and the cleaned prompts; the conventional rule keeps the highest-scored candidate, and `predict_mask` applies exactly that rule for every corpus record. The per-image `evaluation_report` against the self-drawn rectangle is `sample-sanity` — plumbing evidence, not a measurement; whether the model is *good at real targets* is what Section 6 measures on 70 ADE20K regions. The inference-only card recorded scores `[0.955, 1.012, 0.979]` and `mask_iou` 1.000 on this scene.

In [ ]:
scene = Image.new('RGB', (320, 240), (128, 128, 128))
draw = ImageDraw.Draw(scene)
rectangle_box = [40, 60, 140, 180]
draw.rectangle(rectangle_box, fill=(30, 30, 30))
draw.ellipse([200, 80, 280, 160], fill=(220, 30, 30))
scene_reference = np.zeros((240, 320), dtype=bool)
scene_reference[60:181, 40:141] = True
scene_points, scene_labels = [[90, 120]], [1]
scene_name = 'synthetic_scene_320x240.png'
scene_sha256 = hashlib.sha256(np.asarray(scene).tobytes()).hexdigest()
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'NUM_MULTIMASK_OUTPUTS': NUM_MULTIMASK_OUTPUTS, 'MASK_THRESHOLD': MASK_THRESHOLD, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device}})
input_manifest = validate_inputs(scene, points=scene_points, point_labels=scene_labels, multimask=True, names=[scene_name])
try:
    validate_inputs(scene, points=[[scene.width, scene.height]], point_labels=[1])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'click-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/sam_vit_segmentation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene': scene_name, 'sha256': scene_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def segment_scene(pipeline, label):
    started = time.perf_counter()
    result = pipeline.segment(scene, points=scene_points, point_labels=scene_labels, multimask=True)
    elapsed = time.perf_counter() - started
    masks = result['masks']
    best = int(np.argmax(result['iou_scores']))
    checks = {
        'shape': masks.shape == (NUM_MULTIMASK_OUTPUTS, scene.height, scene.width),
        'dtype_bool': masks.dtype == np.bool_,
        'one_score_per_mask': len(result['iou_scores']) == masks.shape[0],
        'identity_reported': result['model_id'] == MODEL_ID and result['model_revision'] == MODEL_REVISION,
    }
    if not all(checks.values()):
        raise RuntimeError(f'segment output failed a sanity check: {checks}')
    report = evaluation_report(result, scene_reference, sample_kind='synthetic (authored in this notebook)')
    Image.fromarray(masks[best]).save(f'outputs/sam_vit_segmentation_mask_{label}.png')
    print({label: {'seconds': round(elapsed, 3), 'checks': checks, 'iou_scores_model_predicted': [round(v, 4) for v in result['iou_scores']], 'scores_above_one': [i for i, v in enumerate(result['iou_scores']) if v > 1.0], 'best_candidate': best, 'mask_iou_vs_reference': {m['candidate']: round(m['value'], 3) for m in report['metrics']} if report['metrics'] else None, 'verdict': report['verdict']}})
    return result, report


frozen_scene_result, frozen_scene = segment_scene(pipe, 'frozen')

## 6. Baselines and the frozen model on the test targets

Two prompt-only baselines frame the adaptation, each scored by `segmentation_metrics` (carried in `metrics.py`): mean **IoU** against the target mask and the **hit rate** — the fraction of targets reaching IoU ≥ 0.5 — overall and per class. The **box-fill** baseline answers with every pixel inside the box: no model, and for a rectangular region the perfect answer. The **centre-disk** baseline draws a disk around the click with the target's own area — the size given away, the shape not. The **frozen model** is scored twice by `pipe.evaluate`: with the **point** prompt (one click, the model's own best-of-three candidate) and with the **box** prompt. Expect the frozen point prompt **below box-fill** — the build record measured 0.564 (hit rate 0.60) against 0.614 — because a single click on a wall or a floor is ambiguous to a model trained to return *an* object, while the box prompt, which states the extent, reaches 0.784. Read the per-class rows to see where the click fails: large stuff regions, not small things.

In [ ]:
METRICS = ('iou', 'hit_rate')
baseline_box = box_fill_baseline(test_records)
baseline_disk = centre_disk_baseline(test_records)
print({'box_fill_baseline': {k: round(baseline_box[k], 3) for k in METRICS}, 'n': baseline_box['n'], 'note': baseline_box['baseline']})
print({'centre_disk_baseline': {k: round(baseline_disk[k], 3) for k in METRICS}, 'note': baseline_disk['baseline']})
t0 = time.perf_counter()
frozen_point = pipe.evaluate(test_records, prompt='point')
frozen_box = pipe.evaluate(test_records, prompt='box')
print({'frozen_point_test': {k: round(frozen_point[k], 3) for k in METRICS}, 'frozen_box_test': {k: round(frozen_box[k], 3) for k in METRICS}, 'n': frozen_point['n'], 'verdict': frozen_point['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_point['definitions'], 'hit_threshold': HIT_THRESHOLD})
frozen_fields = {c: {'n': v['n'], 'point_iou': round(v['iou'], 3), 'box_iou': round(frozen_box['per_category'][c]['iou'], 3)} for c, v in frozen_point['per_category'].items()}
print({'by_class_frozen': dict(sorted(frozen_fields.items(), key=lambda kv: -kv[1]['n']))})
assert frozen_box['iou'] > baseline_disk['iou']

## 7. Bounded fine-tuning of the mask decoder

`pipe.adapt` trains only the **mask decoder** — the two-way transformer, the up-scaling head and the IoU-prediction head, 4,058,340 of 93,735,472 parameters — while the image encoder and the prompt encoder stay frozen. Because the image encoder is frozen, every training image is embedded **once** (cached under `torch.no_grad`, about 30 s on the build GPU) and each step runs only the prompt encoder and the decoder: one record per step, the prompt drawn by a seeded coin — the record's click or its box (`PROMPTS = 'mixed'`) — and the decoder's single-mask logits compared with the target in the 256×256 low-resolution frame under **binary cross-entropy plus a soft Dice term**. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler. Epoch 0 records the frozen model's validation metrics; every epoch is scored on the 45 validation targets with the point prompt, and the epoch with the highest validation point IoU is kept.

Watch the training loss fall from about 0.5 while the validation point IoU climbs by roughly 0.15 over six epochs: the decoder is learning *which* of its candidate granularities this dataset means by a click, which 180 targets are enough to teach. The build record's counter-example — the same recipe at twice the rate for four epochs — gained less on the point prompt and cost the box prompt a tenth; the default is the configuration that lifted the click past both baselines while holding the box.

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
PROMPTS = 'mixed'  # @param ["mixed", "point", "box"]


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_point_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, prompts=PROMPTS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'embedding_seconds': adapt_result['embedding_seconds'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test targets were never used for training or epoch selection, and no image appears in two splits. The adapted decoder is scored exactly as the frozen one was in Section 6 — point prompt and box prompt — the four systems are put side by side, and the per-class breakdown is repeated. Read it in this order: the **point-prompt IoU** first (the measure the epoch was selected on — the build record measured 0.564 → 0.709, past box-fill's 0.614 and the disk's 0.442; hit rate 0.60 → 0.79), then the **box-prompt IoU** (0.784 → 0.772: held within a hundredth, the cost of teaching the click), then the per-class rows, where the large stuff classes gain the most. The cell asserts the adapted point IoU is above the frozen one and reports whether it is above box-fill. Seventy targets from one seeded split of one dataset give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on ADE20K's whole-region convention says nothing about a click on *your* objects until you measure it.

In [ ]:
adapted_point = pipe.evaluate(test_records, prompt='point')
adapted_box = pipe.evaluate(test_records, prompt='box')
adapted_val = pipe.evaluate(val_records, prompt='point')
adapted_fields = {c: {'n': v['n'], 'point_iou': round(v['iou'], 3), 'box_iou': round(adapted_box['per_category'][c]['iou'], 3)} for c, v in adapted_point['per_category'].items()}
comparison = {metric: {'centre_disk': round(baseline_disk[metric], 3), 'box_fill': round(baseline_box[metric], 3), 'frozen_point': round(frozen_point[metric], 3), 'adapted_point': round(adapted_point[metric], 3), 'frozen_box': round(frozen_box[metric], 3), 'adapted_box': round(adapted_box[metric], 3)} for metric in METRICS}
comparison['delta_point_vs_frozen'] = {metric: round(adapted_point[metric] - frozen_point[metric], 3) for metric in METRICS}
comparison['delta_point_vs_box_fill'] = {metric: round(adapted_point[metric] - baseline_box[metric], 3) for metric in METRICS}
comparison['delta_box_vs_frozen'] = {metric: round(adapted_box[metric] - frozen_box[metric], 3) for metric in METRICS}
comparison['by_class'] = {c: {'n': frozen_fields[c]['n'], 'frozen_point': frozen_fields[c]['point_iou'], 'adapted_point': adapted_fields[c]['point_iou'], 'frozen_box': frozen_fields[c]['box_iou'], 'adapted_box': adapted_fields[c]['box_iou']} for c in sorted(frozen_fields, key=lambda c: -frozen_fields[c]['n'])}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'target_rule': {'area_fraction': [TARGET_MIN_FRACTION, TARGET_MAX_FRACTION], 'point': 'distance-transform maximum', 'box': 'tight bounds'},
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'box_fill': {k: v for k, v in baseline_box.items() if k != 'per_record'}, 'centre_disk': {k: v for k, v in baseline_disk.items() if k != 'per_record'}},
    'frozen_test': {'point': frozen_point, 'box': frozen_box},
    'validation_metrics': adapted_val,
    'test_metrics': {'point': adapted_point, 'box': adapted_box},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/sam_vit_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_point['iou'] > frozen_point['iou']
print({'report': 'outputs/sam_vit_segmentation_evaluation_report.json', 'adapted_point_beats_box_fill': adapted_point['iou'] > baseline_box['iou']})

## 9. Look at the masks, export the adapter and reload it

The synthetic scene from Section 5 is segmented again by the adapted decoder — an image family the adaptation never saw, so this is a small look at what it did *outside* its corpus: the rectangle is a crisp thing, not ADE20K stuff, and the click should still return it (the build record kept `mask_iou` above 0.99) — and four held-out targets are written as side-by-side panels (`outputs/sam_vit_segmentation_examples/`: image with the click and the box drawn, frozen point mask, adapted point mask, target) so the numbers can be checked by eye: the adapted panels should cover the whole region where the frozen ones stopped at a part.

`pipe.save_artifact` writes the trained tensors — the mask decoder, about 16 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `SAMViTSegmentationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the mask decoder, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical point-prompt masks on eight test targets (VER4).

In [ ]:
import shutil

adapted_scene_result, adapted_scene = segment_scene(pipe, 'adapted')
examples_dir = Path('outputs/sam_vit_segmentation_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
frozen_base = SAMViTSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)


def tint(image, mask, colour):
    array = np.asarray(image.convert('RGB')).copy()
    array[mask] = (0.45 * array[mask] + 0.55 * np.array(colour)).astype(np.uint8)
    return Image.fromarray(array, mode='RGB')


for record in test_records[:4]:
    image = record['image']
    prompted = image.copy()
    marker = ImageDraw.Draw(prompted)
    marker.rectangle(record['box'], outline=(255, 220, 0), width=3)
    x, y = record['point']
    marker.ellipse([x - 6, y - 6, x + 6, y + 6], fill=(255, 0, 0))
    panels = [prompted, tint(image, frozen_base.predict_mask(record), (0, 120, 255)), tint(image, pipe.predict_mask(record), (0, 200, 80)), tint(image, record['mask'], (255, 255, 255))]
    sheet = Image.new('RGB', (image.width * 4 + 30, image.height), (255, 255, 255))
    for i, panel in enumerate(panels):
        sheet.paste(panel, (i * (image.width + 10), 0))
    sheet.save(examples_dir / f"{record['id']}_{record.get('category', 'target').replace(' ', '-')}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'panel_order': ['image with click and box', 'frozen point mask', 'adapted point mask', 'target']})

artifact_dir = Path('outputs/sam_vit_segmentation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'sam_vit_segmentation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = SAMViTSegmentationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.predict_mask(r) for r in test_records[:8]]
after = [reloaded.predict_mask(r) for r in test_records[:8]]
parity = {'identical_masks': sum(np.array_equal(a, b) for a, b in zip(before, after, strict=True)), 'of': len(before), 'max_pixels_differing': int(max((a != b).sum() for a, b in zip(before, after, strict=True)))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_masks'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'shard_bytes': CORPUS_BYTES, 'row_groups': {str(k): {'sha256': v[0], 'bytes': v[1]} for k, v in ROW_GROUP_PINS.items()}, 'target_rule': {'area_fraction': [TARGET_MIN_FRACTION, TARGET_MAX_FRACTION]}},
    'inference_contract': {'input_manifest': input_manifest, 'scene': {'name': scene_name, 'sha256': scene_sha256}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene, 'output_files': ['outputs/sam_vit_segmentation_mask_frozen.png', 'outputs/sam_vit_segmentation_mask_adapted.png']},
    'comparison': comparison,
    'examples': 'outputs/sam_vit_segmentation_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/sam_vit_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A single click on an ADE20K region is ambiguous to the frozen SAM decoder — it returns a part, or the region with its neighbours, often enough that the mean IoU (0.564 in the build record) sits below simply filling the box (0.614) — and a bounded fine-tuning of the 4-million-parameter mask decoder on 180 targets under mixed prompts resolves the ambiguity the way this dataset does (point IoU 0.709, hit rate 0.60 → 0.79) while holding the box prompt within a hundredth (0.784 → 0.772), with a 16 MB adapter that reloads mask-for-mask. That is the claim: the adaptation contract works end to end on a real labelled set, and the numbers it produces are read on two prompts, per class, against two prompt-only baselines and the frozen model rather than in isolation.

The test split is 70 targets from one seeded draw of the first 300 validation images, the validation split that picks the epoch is 45, and the targets were chosen by a rule (largest component, 3..35 % of the image) that favours large stuff regions — walls, sky, floors, roads — over the small things a click is usually for. So a gain here says the decoder learned ADE20K's *whole-region* reading of a click, not that it will read your click the way you mean it, that it handles thin or occluded objects, or that its predicted IoU is now calibrated (it is not; scores above 1.0 still occur). The adapted decoder is also a different model outside its corpus: the synthetic rectangle in Section 9 is one image of evidence that crisp things survive, not a measurement.

Three things to carry to real data. **Baselines first:** fill the box and draw the disk on *your* targets before reading any model number, per class. **Labelling convention:** the masks the decoder learns from define what a click means; label your targets at the granularity you want returned, and keep the box prompt in the evaluation so a click-only gain that costs the box is visible. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels) and split by scene or session when your images come from few sources.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a slice of a real segmentation dataset and turn it into prompt-to-mask targets under a stated rule, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two prompt-only baselines and the frozen model on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, segmentation quality on any other labelling convention, calibration of the predicted IoU, or production fitness.

**Optional experiments (they do not affect the default path):** set `PROMPTS = 'point'` and read whether the click gains more while the box loses more; set `PROMPTS = 'box'` and watch the point prompt barely move; raise `EPOCHS` and watch the validation IoU pick the epoch while the loss keeps falling; change `LEARNING_RATE` to `1e-4` and read the faster, box-costing climb the build record measured; or bring your own masks through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/sam-vit-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/sam-vit-base
- Upstream code: https://github.com/facebookresearch/segment-anything
- Segment Anything (Kirillov et al., ICCV 2023): https://arxiv.org/abs/2304.02643
- ADE20K scene parsing (Zhou et al., CVPR 2017; IJCV 2019; BSD-3-Clause): https://github.com/CSAILVision/sceneparsing — parquet conversion https://huggingface.co/datasets/zhoubolei/scene_parse_150
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)